# Loss-Based Pipeline (Separate Notebook)

This notebook keeps the original perplexity notebook unchanged and runs a parallel analysis using avg_loss_completion.

In [13]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from services.analysis_service import load_metadata
from functions.analysis.statistics import *

In [14]:
TYPE = "prompt_healthy"
MODEL = "qwen"
SENTENCE_CSV = f"data/02_perplexities/Qwen2.5-32B-Instruct/{TYPE}/TAT_german_utterances_with_perplexities_{TYPE.split('_', 1)[1]}.csv"
METADATA_CSV = "data/00_raw_data/speech_metadata.csv"
MATCH_VARS = [f"{MODEL}_avg_words", "education", "age", "sex"]
MATCH_K = 2
MAIN_VAR = f"{MODEL}_loss_resid_median"

In [3]:
sentence_df = pd.read_csv(SENTENCE_CSV)
metadata = load_metadata(METADATA_CSV)

required = ["file_id", "avg_loss_completion", "word_count_completion", "utterance"]
missing = [c for c in required if c not in sentence_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

sentence_df_resid, length_model = add_length_residualized_loss(sentence_df)
agg_loss = aggregate_loss_completion(sentence_df_resid, MODEL)
agg_resid = aggregate_residualized_loss(sentence_df_resid, MODEL)
agg_ppl = aggregate_perplexity(sentence_df_resid, MODEL)

merge_keys = ["file_id", f"{MODEL}_total_words", f"{MODEL}_avg_words", f"{MODEL}_n_sentences"]
person_df = (
    agg_loss
    .merge(agg_resid, on=merge_keys, how="outer")
    .merge(agg_ppl, on=merge_keys, how="outer")
)
df = metadata.merge(person_df, on="file_id", how="inner")

print(df.shape)
display(df.head())

(366, 45)


,file_id,diagnosis,sex,age,education,comorbidity,GAF,TLI_Impoverished,TLI_Disorganization,verbal_IQ,...,qwen_loss_resid_mean,qwen_loss_resid_median,qwen_loss_resid_std,qwen_loss_resid_p90,qwen_ppl_mean,qwen_ppl_median,qwen_ppl_std,qwen_ppl_p90,qwen_ppl_p75,qwen_ppl_cv
0,0006_3,MDD,male,30,13.0,0,65.0,0.0,2.25,NaN,...,-0.003438,0.031294,0.840638,0.910132,96.335367,42.502467,254.154028,150.596429,79.299858,2.638221
1,0008_3,HC,female,24,13.0,0,82.0,1.0,0.25,107.0,...,-0.529404,-0.485244,0.877589,0.526815,46.603439,27.029868,52.406966,97.844981,54.298017,1.124530
2,0011_3,MDD,male,52,9.0,1,79.0,0.0,1.00,94.0,...,0.224909,0.236925,0.842665,1.254361,123.540934,59.887345,170.630051,300.983178,130.207014,1.381162
3,0013_3,MDD,female,44,13.0,0,75.0,0.0,0.25,118.0,...,0.147060,0.058796,0.979941,1.296772,285.384255,43.634203,1563.709635,262.746527,101.114324,5.479313
4,0020_3,MDD,female,26,12.0,1,45.0,0.0,0.25,104.0,...,0.016057,-0.169496,1.054308,1.047274,381.086769,46.186935,2284.354791,283.994930,111.010375,5.994317


In [29]:
loss_cols = [
    f"{MODEL}_loss_mean",
    f"{MODEL}_loss_median",
    f"{MODEL}_loss_resid_mean",
    f"{MODEL}_loss_resid_median",
]
loss_cols = [c for c in loss_cols if c in df.columns]

settings = {}
settings["with_outliers"] = df.copy()
settings["without_outliers"] = winsorize_and_log(df, loss_cols) if loss_cols else df.copy()
settings["matched"] = create_matched_sample(df, match_vars=MATCH_VARS)
settings["matched_without_outliers"] = winsorize_and_log(settings["matched"], [c for c in loss_cols if c in settings["matched"].columns]) if loss_cols else settings["matched"].copy()
settings["matched_per_dx"] = create_matched_sample_per_diagnosis(df, match_vars=MATCH_VARS, k=MATCH_K)
settings["matched_per_dx_without_outliers"] = winsorize_and_log(settings["matched_per_dx"], [c for c in loss_cols if c in settings["matched_per_dx"].columns]) if loss_cols else settings["matched_per_dx"].copy()

for name, sdf in settings.items():
    n_pat = (sdf["is_patient"] == 1).sum()
    n_hc = (sdf["is_patient"] == 0).sum()
    print(f"{name}: {len(sdf)} rows ({n_pat} patients, {n_hc} controls)")

  Matched sample: 158 patients + 158 HC  (match_vars=['qwen_avg_words', 'education', 'age', 'sex'])
  BD: 22 patients × up to 2 HCs → 44 matched HC rows (44 unique)
  MDD: 83 patients × up to 2 HCs → 162 matched HC rows (162 unique)
  SSD: 39 patients × up to 2 HCs → 77 matched HC rows (77 unique)
  Per-diagnosis matched total: 144 patients + 283 HC rows  (match_vars=['qwen_avg_words', 'education', 'age', 'sex'], k=2)
with_outliers: 366 rows (189 patients, 177 controls)
without_outliers: 366 rows (189 patients, 177 controls)
matched: 316 rows (158 patients, 158 controls)
matched_without_outliers: 316 rows (158 patients, 158 controls)
matched_per_dx: 427 rows (144 patients, 283 controls)
matched_per_dx_without_outliers: 427 rows (144 patients, 283 controls)


In [30]:
clinical_vars_corr = [v for v in CLINICAL_VARS if v in df.columns]
rows = []

for setting_name, sdf in settings.items():
    var = MAIN_VAR
    if setting_name.endswith("without_outliers") and f"{MAIN_VAR}_wins" in sdf.columns:
        var = f"{MAIN_VAR}_wins"
    if var not in sdf.columns:
        print(f"Skipping {setting_name}: {var} missing")
        continue

    comp = group_comparison(sdf, var)
    rows.append({
        "setting": setting_name,
        "variable": var,
        "p_mw": comp["p(MW)"],
        "cohens_d": comp["Cohens d"],
        "control_median": comp["Control Median"],
        "patient_median": comp["Patient Median"],
    })

    corr = clinical_correlations(sdf[sdf["is_patient"] == 1], var, clinical_vars_corr)
    out_dir = Path("figures") / "loss_pipeline" / setting_name
    out_dir.mkdir(parents=True, exist_ok=True)
    corr.to_csv(out_dir / "clinical_correlations.csv", index=False)

summary_df = pd.DataFrame(rows)
display(summary_df)

root_out = Path("figures") / "loss_pipeline"
root_out.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(root_out / "summary_group_comparison.csv", index=False)
df.to_csv(root_out / "person_level_loss_features.csv", index=False)
print(f"Saved outputs to: {root_out.resolve()}")

,setting,variable,p_mw,cohens_d,control_median,patient_median
0,with_outliers,qwen_loss_resid_median,0.110562,0.132188,-0.049961,0.005687
1,without_outliers,qwen_loss_resid_median_wins,0.109852,0.152618,-0.049961,0.005687
2,matched,qwen_loss_resid_median,0.138667,0.145980,-0.086216,-0.036820
3,matched_without_outliers,qwen_loss_resid_median_wins,0.136820,0.165935,-0.086216,-0.036820
4,matched_per_dx,qwen_loss_resid_median,0.002017,0.285272,-0.090442,0.023594
5,matched_per_dx_without_outliers,qwen_loss_resid_median_wins,0.001994,0.314449,-0.090442,0.023594


,setting,variable,p_mw,cohens_d,control_median,patient_median
0,with_outliers,qwen_loss_resid_median,0.110562,0.132188,-0.049961,0.005687
1,without_outliers,qwen_loss_resid_median_wins,0.109852,0.152618,-0.049961,0.005687
2,matched,qwen_loss_resid_median,0.138667,0.145980,-0.086216,-0.036820
3,matched_without_outliers,qwen_loss_resid_median_wins,0.136820,0.165935,-0.086216,-0.036820
4,matched_per_dx,qwen_loss_resid_median,0.002017,0.285272,-0.090442,0.023594
5,matched_per_dx_without_outliers,qwen_loss_resid_median_wins,0.001994,0.314449,-0.090442,0.023594


Saved outputs to: /zi/home/esra.lenz/Documents/00_HITKIP/11_normative/mnm_ppl/figures/loss_pipeline


In [ ]:
# Robustness checks: is sentence-level residualization over-correcting?
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import statsmodels.formula.api as smf


def cohens_d(g1, g0):
    g1 = pd.to_numeric(g1, errors="coerce").dropna()
    g0 = pd.to_numeric(g0, errors="coerce").dropna()
    if len(g1) < 2 or len(g0) < 2:
        return np.nan
    pooled = np.sqrt(((len(g1) - 1) * g1.std() ** 2 + (len(g0) - 1) * g0.std() ** 2) / (len(g1) + len(g0) - 2))
    return (g1.mean() - g0.mean()) / pooled if pooled > 0 else np.nan


def mw_compare(df_in, var, label):
    sub = df_in[["is_patient", var]].dropna().copy()
    g0 = sub[sub["is_patient"] == 0][var]
    g1 = sub[sub["is_patient"] == 1][var]
    if len(g0) < 2 or len(g1) < 2:
        return {"setting": label, "variable": var, "n_hc": len(g0), "n_pat": len(g1), "p_mw": np.nan, "d": np.nan, "hc_median": np.nan, "pat_median": np.nan}
    _, p = mannwhitneyu(g1, g0, alternative="two-sided")
    d = cohens_d(g1, g0)
    return {
        "setting": label,
        "variable": var,
        "n_hc": len(g0),
        "n_pat": len(g1),
        "p_mw": p,
        "d": d,
        "hc_median": g0.median(),
        "pat_median": g1.median(),
    }


rows = []

# 1) Raw loss median vs residualized loss median across settings
for setting_name, sdf in settings.items():
    if f"{MODEL}_loss_median" in sdf.columns:
        rows.append(mw_compare(sdf, f"{MODEL}_loss_median", f"{setting_name} | raw_loss_median"))
    if MAIN_VAR in sdf.columns:
        rows.append(mw_compare(sdf, MAIN_VAR, f"{setting_name} | resid_loss_median"))
    if f"{MAIN_VAR}_wins" in sdf.columns:
        rows.append(mw_compare(sdf, f"{MAIN_VAR}_wins", f"{setting_name} | resid_loss_median_wins"))

robust_df = pd.DataFrame(rows)
display(robust_df.sort_values(["setting", "variable"]).reset_index(drop=True))

# 2) Check duplicate file_id inflation in per-diagnosis matched settings and re-test on unique subjects
for sname in ["matched_per_dx", "matched_per_dx_without_outliers"]:
    if sname not in settings:
        continue
    sdf = settings[sname].copy()
    if "file_id" not in sdf.columns:
        print(f"{sname}: no file_id column; cannot deduplicate")
        continue

    n_total = len(sdf)
    n_unique = sdf["file_id"].nunique()
    dup_rows = n_total - n_unique
    print(f"\n{sname}: total rows={n_total}, unique file_id={n_unique}, duplicated rows={dup_rows}")

    # Deduplicate by file_id (keep first row for each person)
    sdf_u = sdf.sort_values("file_id").drop_duplicates(subset=["file_id"], keep="first")

    test_vars = [v for v in [f"{MODEL}_loss_median", MAIN_VAR, f"{MAIN_VAR}_wins"] if v in sdf_u.columns]
    for v in test_vars:
        out_full = mw_compare(sdf, v, f"{sname} | full")
        out_uniq = mw_compare(sdf_u, v, f"{sname} | unique_file_id")
        print(f"  {v}: p(full)={out_full['p_mw']:.4g}, p(unique)={out_uniq['p_mw']:.4g}, d(full)={out_full['d']:.3f}, d(unique)={out_uniq['d']:.3f}")

# 3) Optional sentence-level mixed model (keeps group effect explicit while adjusting length)
# Requires the original sentence_df + metadata, already loaded earlier in this notebook.
if "sentence_df" in globals() and "metadata" in globals():
    sent = sentence_df.copy()
    needed = ["file_id", "avg_loss_completion", "word_count_completion"]
    if all(c in sent.columns for c in needed):
        mdf = metadata[["file_id", "is_patient"]].drop_duplicates()
        sent = sent.merge(mdf, on="file_id", how="inner")
        sent["avg_loss_completion"] = pd.to_numeric(sent["avg_loss_completion"], errors="coerce")
        sent["word_count_completion"] = pd.to_numeric(sent["word_count_completion"], errors="coerce")
        sent["log_wc"] = np.log1p(sent["word_count_completion"].clip(lower=0))
        mix = sent[["file_id", "is_patient", "avg_loss_completion", "log_wc"]].dropna().copy()

        if len(mix) > 50 and mix["is_patient"].nunique() == 2 and mix["file_id"].nunique() > 20:
            try:
                model = smf.mixedlm("avg_loss_completion ~ is_patient + log_wc", data=mix, groups=mix["file_id"])
                fit = model.fit(reml=False, method="lbfgs")
                print("\nMixed model fitted: avg_loss_completion ~ is_patient + log_wc + (1|file_id)")
                display(fit.summary())
            except Exception as e:
                print(f"\nMixed model could not be fit: {e}")
        else:
            print("\nNot enough data variation for stable mixed model.")
    else:
        print("\nSentence-level columns needed for mixed model are missing.")
else:
    print("\nRun earlier cells first so sentence_df and metadata are available for mixed model.")

,setting,variable,n_hc,n_pat,p_mw,d,hc_median,pat_median
0,matched | raw_loss_median,qwen_loss_median,158,158,0.010745,0.260213,3.672633,3.765869
1,matched | resid_loss_median,qwen_loss_resid_median,158,158,0.138667,0.145980,-0.086216,-0.036820
2,matched_per_dx | raw_loss_median,qwen_loss_median,283,144,0.000230,0.367733,3.680230,3.832433
3,matched_per_dx | resid_loss_median,qwen_loss_resid_median,283,144,0.002017,0.285272,-0.090442,0.023594
4,matched_per_dx_without_outliers | raw_loss_median,qwen_loss_median,283,144,0.000230,0.367733,3.680230,3.832433
5,matched_per_dx_without_outliers | resid_loss_m...,qwen_loss_resid_median,283,144,0.002017,0.285272,-0.090442,0.023594
6,matched_per_dx_without_outliers | resid_loss_m...,qwen_loss_resid_median_wins,283,144,0.001994,0.314449,-0.090442,0.023594
7,matched_without_outliers | raw_loss_median,qwen_loss_median,158,158,0.010745,0.260213,3.672633,3.765869
8,matched_without_outliers | resid_loss_median,qwen_loss_resid_median,158,158,0.138667,0.145980,-0.086216,-0.036820
9,matched_without_outliers | resid_loss_median_wins,qwen_loss_resid_median_wins,158,158,0.136820,0.165935,-0.086216,-0.036820



matched_per_dx: total rows=427, unique file_id=306, duplicated rows=121
  qwen_loss_median: p(full)=0.0002303, p(unique)=0.0002503, d(full)=0.368, d(unique)=0.395
  qwen_loss_resid_median: p(full)=0.002017, p(unique)=0.00845, d(full)=0.285, d(unique)=0.274

matched_per_dx_without_outliers: total rows=427, unique file_id=306, duplicated rows=121
  qwen_loss_median: p(full)=0.0002303, p(unique)=0.0002503, d(full)=0.368, d(unique)=0.395
  qwen_loss_resid_median: p(full)=0.002017, p(unique)=0.00845, d(full)=0.285, d(unique)=0.274
  qwen_loss_resid_median_wins: p(full)=0.001994, p(unique)=0.008343, d(full)=0.314, d(unique)=0.302

Mixed model fitted: avg_loss_completion ~ is_patient + log_wc + (1|file_id)


/zi/home/esra.lenz/Documents/00_HITKIP/11_normative/mnm_ppl/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/zi/home/esra.lenz/Documents/00_HITKIP/11_normative/mnm_ppl/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/zi/home/esra.lenz/Documents/00_HITKIP/11_normative/mnm_ppl/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/zi/home/esra.lenz/Documents/00_HITKIP/11_normative/mnm_ppl/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/zi/home/esra.lenz/Documents/00_HITKIP

<class 'statsmodels.iolib.summary2.Summary'>
"""
               Mixed Linear Model Regression Results
===================================================================
Model:              MixedLM Dependent Variable: avg_loss_completion
No. Observations:   26514   Method:             ML                 
No. Groups:         366     Scale:              0.9530             
Min. group size:    10      Log-Likelihood:     inf                
Max. group size:    199     Converged:          Yes                
Mean group size:    72.4                                           
-------------------------------------------------------------------
           Coef.   Std.Err.     z    P>|z|    [0.025       0.975]  
-------------------------------------------------------------------
Intercept  -0.000 389066.022  -0.000 1.000  -762555.391  762555.391
is_patient -0.000 740569.378  -0.000 1.000 -1451489.309 1451489.309
log_wc     -0.765      0.008 -97.388 0.000       -0.780      -0.749
Group Var   0.000                                                  
===================================================================

"""

In [15]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, balanced_accuracy_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer


def compare_random_forest_incremental(
    data: pd.DataFrame,
    y,
    confound_vars: list,
    length_vars: list,
    ppl_vars: list,
    n_splits: int = 5,
    random_state: int = 42,
):
    """
    Compare baseline vs full Random Forest models.

    Baseline:
        y ~ confounds + length

    Full:
        y ~ confounds + length + perplexity/loss variables

    Returns cross-validated performance difference.
    """

    X_base = data[confound_vars + length_vars].astype(float)
    X_full = data[confound_vars + length_vars + ppl_vars].astype(float)

    y = np.asarray(y)

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    rf = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=-1,
    )

    pipe = make_pipeline(
        SimpleImputer(strategy="median"),
        rf,
    )

    scoring = {
        "auc": "roc_auc",
        "balanced_accuracy": make_scorer(balanced_accuracy_score),
    }

    base_scores = cross_validate(
        pipe,
        X_base,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
    )

    full_scores = cross_validate(
        pipe,
        X_full,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
    )

    delta_auc = full_scores["test_auc"] - base_scores["test_auc"]
    delta_bal_acc = (
        full_scores["test_balanced_accuracy"]
        - base_scores["test_balanced_accuracy"]
    )

    result = {
        "features_added": ppl_vars,
        "auc_baseline_mean": float(np.mean(base_scores["test_auc"])),
        "auc_baseline_sd": float(np.std(base_scores["test_auc"])),
        "auc_full_mean": float(np.mean(full_scores["test_auc"])),
        "auc_full_sd": float(np.std(full_scores["test_auc"])),
        "delta_auc_mean": float(np.mean(delta_auc)),
        "delta_auc_folds": delta_auc.tolist(),
        "balanced_acc_baseline_mean": float(
            np.mean(base_scores["test_balanced_accuracy"])
        ),
        "balanced_acc_full_mean": float(
            np.mean(full_scores["test_balanced_accuracy"])
        ),
        "delta_balanced_acc_mean": float(np.mean(delta_bal_acc)),
        "delta_balanced_acc_folds": delta_bal_acc.tolist(),
    }

    return result

In [16]:
import numpy as np
import pandas as pd
from scipy.stats import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm


def perplexity_value_added(
    df,
    outcome,
    length_vars,
    ppl_vars,
    confound_vars=None,
    n_splits=5,
    n_repeats=20,
    n_boot=2000,
    group_col=None,
    random_state=42,
):
    """Test whether perplexity / loss adds predictive value beyond length
    (and optionally beyond demographic confounders).

    Three logistic models are fit under repeated stratified CV — or
    GroupKFold if `group_col` is given so duplicate subjects stay in the
    same fold:

        length_only : confound + length
        ppl_only    : confound + ppl
        combined    : confound + length + ppl

    Reports per-model AUC + 95% CI, a bootstrap CI on the ΔAUC
    (combined - length_only) over OOF predictions, and a likelihood-ratio
    test of the same nested logits on the full sample. When `group_col`
    is set, the bootstrap resamples groups (not rows) so the CI accounts
    for within-subject duplication.

    Parameters
    ----------
    df            : DataFrame with all needed columns
    outcome       : binary outcome column (e.g. "is_patient")
    length_vars   : list of length predictors
    ppl_vars      : list of perplexity / loss predictors
    confound_vars : list of demographic / nuisance predictors
                    (e.g. ["education", "sex_num", "age"]).
                    These are added to ALL three feature sets so every
                    comparison is "above and beyond confounders".
    group_col     : optional grouping column (e.g. "file_id" for
                    matched_per_dx settings with duplicate subjects).

    Returns
    -------
    dict with keys:
        auc_summary, delta_auc, lrt, oof_predictions, y_true,
        feature_sets (the actual columns used in each model).
    """
    rng = np.random.default_rng(random_state)
    confound_vars = list(confound_vars or [])
    length_vars = list(length_vars)
    ppl_vars = list(ppl_vars)

    needed = list(dict.fromkeys(
        confound_vars + length_vars + ppl_vars + [outcome]
        + ([group_col] if group_col else [])
    ))
    data = df[needed].dropna().reset_index(drop=True).copy()
    y = data[outcome].astype(int).to_numpy()
    groups = data[group_col].to_numpy() if group_col else None

    feature_sets = {
        "length_only": confound_vars + length_vars,
        "ppl_only":    confound_vars + ppl_vars,
        "combined":    confound_vars + length_vars + ppl_vars,
    }

    n = len(data)
    oof_proba = {name: np.zeros(n) for name in feature_sets}
    oof_count = np.zeros(n)
    fold_aucs = {name: [] for name in feature_sets}

    for rep in range(n_repeats):
        if group_col is not None:
            uniq = np.array(pd.unique(pd.Series(groups)))
            perm = rng.permutation(len(uniq))
            order_map = {g: i for i, g in enumerate(uniq[perm])}
            shuffled = np.array([order_map[g] for g in groups])
            split_iter = GroupKFold(n_splits=n_splits).split(data, y, groups=shuffled)
        else:
            split_iter = StratifiedKFold(
                n_splits=n_splits, shuffle=True,
                random_state=int(rng.integers(1 << 31)),
            ).split(data, y)

        for tr, te in split_iter:
            oof_count[te] += 1
            for name, feats in feature_sets.items():
                if not feats:
                    p = np.full(len(te), y[tr].mean())
                else:
                    pipe = Pipeline([
                        ("scale", StandardScaler()),
                        ("lr",    LogisticRegression(max_iter=2000, solver="lbfgs")),
                    ])
                    pipe.fit(data.iloc[tr][feats].to_numpy(dtype=float), y[tr])
                    p = pipe.predict_proba(data.iloc[te][feats].to_numpy(dtype=float))[:, 1]
                oof_proba[name][te] += p
                if len(np.unique(y[te])) > 1:
                    fold_aucs[name].append(roc_auc_score(y[te], p))

    for name in feature_sets:
        oof_proba[name] /= np.maximum(oof_count, 1)

    auc_summary = pd.DataFrame([
        {
            "model":    name,
            "mean_auc": float(np.mean(a)) if a else np.nan,
            "ci_lo":    float(np.percentile(a, 2.5)) if a else np.nan,
            "ci_hi":    float(np.percentile(a, 97.5)) if a else np.nan,
            "n_folds":  len(a),
        }
        for name, a in fold_aucs.items()
    ])

    # Bootstrap ΔAUC (combined - length_only). Resample groups when given.
    deltas = np.empty(n_boot)
    if group_col is not None:
        uniq = np.array(pd.unique(pd.Series(groups)))
        rows_by_group = {g: np.where(groups == g)[0] for g in uniq}
        for b in range(n_boot):
            sampled = rng.choice(uniq, size=len(uniq), replace=True)
            idx = np.concatenate([rows_by_group[g] for g in sampled])
            if len(np.unique(y[idx])) < 2:
                deltas[b] = np.nan
                continue
            deltas[b] = (
                roc_auc_score(y[idx], oof_proba["combined"][idx])
                - roc_auc_score(y[idx], oof_proba["length_only"][idx])
            )
    else:
        for b in range(n_boot):
            idx = rng.integers(0, n, size=n)
            if len(np.unique(y[idx])) < 2:
                deltas[b] = np.nan
                continue
            deltas[b] = (
                roc_auc_score(y[idx], oof_proba["combined"][idx])
                - roc_auc_score(y[idx], oof_proba["length_only"][idx])
            )
    deltas = deltas[~np.isnan(deltas)]
    delta_auc = {
        "mean":             float(deltas.mean()),
        "ci_lo":            float(np.percentile(deltas, 2.5)),
        "ci_hi":            float(np.percentile(deltas, 97.5)),
        "p_one_sided_gt0":  float((deltas <= 0).mean()),
        "n_boot_valid":     int(len(deltas)),
    }

    # LRT on full sample: confound + length  vs  confound + length + ppl
    X_base = sm.add_constant(data[confound_vars + length_vars].astype(float))
    X_full = sm.add_constant(data[confound_vars + length_vars + ppl_vars].astype(float))
    m_base = sm.Logit(y, X_base).fit(disp=False)
    m_full = sm.Logit(y, X_full).fit(disp=False)
    lr_stat = 2 * (m_full.llf - m_base.llf)
    df_diff = int(m_full.df_model - m_base.df_model)
    lrt = {
        "LR_stat":            float(lr_stat),
        "df":                 df_diff,
        "p":                  float(1 - chi2.cdf(lr_stat, df_diff)),
        "pseudo_r2_baseline": float(m_base.prsquared),
        "pseudo_r2_full":     float(m_full.prsquared),
    }

    rf_result = compare_random_forest_incremental(
        data=data,
        y=y,
        confound_vars=confound_vars,
        length_vars=length_vars,
        ppl_vars=ppl_vars,
        n_splits=5,
        random_state=42,
    )

    return {
    "auc_summary":     auc_summary,
    "delta_auc":       delta_auc,
    "lrt":             lrt,
    "rf_result":       rf_result,
    "oof_predictions": pd.DataFrame(oof_proba, index=data.index),
    "y_true":          y,
    "feature_sets":    feature_sets,
}

In [31]:
res_cf = perplexity_value_added(
    df=settings["with_outliers"],
    outcome="is_patient",
    confound_vars=["education", "sex_num", "age"],
    length_vars=["qwen_total_words"],
    ppl_vars=["qwen_ppl_median"],
)

In [32]:
res_cf

{'auc_summary':          model  mean_auc     ci_lo     ci_hi  n_folds
 0  length_only  0.632183  0.505719  0.761091      100
 1     ppl_only  0.646489  0.539216  0.782586      100
 2     combined  0.640855  0.526491  0.775783      100,
 'delta_auc': {'mean': 0.009798560923687154,
  'ci_lo': -0.016371559533141816,
  'ci_hi': 0.036323920409513884,
  'p_one_sided_gt0': 0.2265,
  'n_boot_valid': 2000},
 'lrt': {'LR_stat': 4.643387115459632,
  'df': 1,
  'p': 0.03117341803932716,
  'pseudo_r2_baseline': 0.05031541899876035,
  'pseudo_r2_full': 0.059893530311248555},
 'rf_result': {'features_added': ['qwen_ppl_median'],
  'auc_baseline_mean': 0.5986450354097413,
  'auc_baseline_sd': 0.03258524540109769,
  'auc_full_mean': 0.6150274606156959,
  'auc_full_sd': 0.027815223550966408,
  'delta_auc_mean': 0.01638242520595461,
  'delta_auc_folds': [0.08681408681408675,
   0.013888888888889062,
   0.006535947712418277,
   -0.030228758169934755,
   0.004901960784313708],
  'balanced_acc_baseline_mean

In [24]:
# without confounders (current behaviour)
res = perplexity_value_added(
    df=settings["with_outliers"],
    outcome="is_patient",
    length_vars=["qwen_total_words"],
    ppl_vars=["qwen_loss_median"],
)

# with demographic confounders — the version a reviewer expects
res_cf = perplexity_value_added(
    df=settings["with_outliers"],
    outcome="is_patient",
    confound_vars=["education", "sex_num", "age"],
    length_vars=["qwen_total_words"],
    ppl_vars=["qwen_loss_median"],
)

# matched_per_dx: avoid subject leakage in CV and bootstrap
res_dx = perplexity_value_added(
    df=settings["matched_per_dx"],
    outcome="is_patient",
    confound_vars=["education", "sex_num", "age"],
    length_vars=["qwen_total_words"],
    ppl_vars=["qwen_loss_median"],
    group_col="file_id",
)

In [17]:
from itertools import combinations
import pandas as pd


def perplexity_value_added_by_diagnosis(
    df,
    diagnosis_col="diagnosis",
    hc_label="HC",
    patient_labels=None,                # auto-detect from df if None
    include_all_patients=True,          # HC vs all patients pooled
    include_hc_vs_each=True,            # HC vs each diagnosis separately
    include_patient_pairwise=False,     # diagnoses against each other
    min_n_per_group=10,                 # skip a contrast if either class < this
    small_class_threshold=25,           # auto-reduce folds if min(n0, n1) < this
    small_n_splits=3,                   # fold count to use for small contrasts
    **kwargs,                            # forwarded to perplexity_value_added
):
    """Run perplexity_value_added across multiple HC- and patient-level contrasts.

    Each contrast filters the dataframe to two diagnosis groups, builds a
    binary outcome (`_y`), and calls perplexity_value_added. For small
    contrasts the CV fold count is auto-reduced.

    Contrast naming convention: "<class0>_vs_<class1>" — the second label
    is encoded as the positive class (y=1). AUC is symmetric so this is
    cosmetic, but coefficient signs in the LRT depend on it.

    Returns
    -------
    dict with keys:
        summary      : tidy DataFrame, one row per (contrast, model)
        by_contrast  : dict mapping contrast name → full result dict
                       (or None if skipped/failed)
    """
    # ── 1. Decide which patient diagnoses to include ─────────────────
    if patient_labels is None:
        patient_labels = sorted(
            d for d in df[diagnosis_col].dropna().unique() if d != hc_label
        )

    # ── 2. Build the contrast registry ───────────────────────────────
    # Each entry: name -> (sub_df, build_y) where build_y(s) returns the
    # binary outcome for that contrast.
    contrasts = {}

    if include_all_patients:
        sub = df.loc[df[diagnosis_col].notna()].copy()
        contrasts["HC_vs_all"] = (
            sub,
            lambda s: (s[diagnosis_col] != hc_label).astype(int),
        )

    if include_hc_vs_each:
        for dx in patient_labels:
            sub = df.loc[df[diagnosis_col].isin([hc_label, dx])].copy()
            contrasts[f"HC_vs_{dx}"] = (
                sub,
                lambda s, target=dx: (s[diagnosis_col] == target).astype(int),
            )

    if include_patient_pairwise:
        for a, b in combinations(patient_labels, 2):
            sub = df.loc[df[diagnosis_col].isin([a, b])].copy()
            contrasts[f"{a}_vs_{b}"] = (
                sub,
                lambda s, target=b: (s[diagnosis_col] == target).astype(int),
            )

    # ── 3. Run perplexity_value_added on each contrast ───────────────
    default_n_splits = kwargs.pop("n_splits", 5)

    results = {}
    rows = []
    for name, (sub, build_y) in contrasts.items():
        sub = sub.copy()
        sub["_y"] = build_y(sub)
        n0 = int((sub["_y"] == 0).sum())
        n1 = int((sub["_y"] == 1).sum())
        n_min = min(n0, n1)

        if n_min < min_n_per_group:
            print(f"\n=== {name}: n_class0={n0}, n_class1={n1} "
                  f"— skipped (< {min_n_per_group} per group)")
            results[name] = None
            continue

        n_splits_used = small_n_splits if n_min < small_class_threshold else default_n_splits
        note = (f"  [folds={n_splits_used}"
                f"{' — auto-reduced' if n_splits_used != default_n_splits else ''}]")
        print(f"\n=== {name}: n_class0={n0}, n_class1={n1} ==={note}")

        try:
            res = perplexity_value_added(
                df=sub, outcome="_y", n_splits=n_splits_used, **kwargs
            )
        except Exception as e:
            print(f"  failed: {e}")
            results[name] = None
            continue

        results[name] = res
        for _, r in res["auc_summary"].iterrows():
            rows.append({
                "contrast":     name,
                "n_class0":     n0,
                "n_class1":     n1,
                "n_splits":     n_splits_used,
                "model":        r["model"],
                "mean_auc":     r["mean_auc"],
                "auc_ci_lo":    r["ci_lo"],
                "auc_ci_hi":    r["ci_hi"],
                "delta_auc":    res["delta_auc"]["mean"],
                "delta_ci_lo": res["delta_auc"]["ci_lo"],
                "delta_ci_hi": res["delta_auc"]["ci_hi"],
                "delta_p_1s":   res["delta_auc"]["p_one_sided_gt0"],
                "lrt_p":        res["lrt"]["p"],
                "pr2_baseline": res["lrt"]["pseudo_r2_baseline"],
                "pr2_full":     res["lrt"]["pseudo_r2_full"],
                "rf_delta_auc": res["rf_result"]["delta_auc_mean"],
            })

    return {"summary": pd.DataFrame(rows), "by_contrast": results}

In [6]:
out = perplexity_value_added_by_diagnosis(
        df=settings["without_outliers"],
        include_all_patients=True,
        include_hc_vs_each=True,
        include_patient_pairwise=True,
        confound_vars=["education", "sex_num", "age"],
        length_vars=["qwen_total_words"],
        ppl_vars=[ppl_vars],
        n_splits=5,                  # default; auto-drops to small_n_splits when n_min < 25
        n_repeats=20,
        min_n_per_group=10,
        small_class_threshold=25,
        small_n_splits=3,
    )
print(out["summary"].to_string(index=False))

NameError: name 'perplexity_value_added_by_diagnosis' is not defined

In [24]:
from scipy.stats import pearsonr, spearmanr
import numpy as np
import pandas as pd

def corr_with_word_count_sentence(sentence_df_resid, model_col="avg_loss_length_resid", length_col="word_count_completion"):
    df = sentence_df_resid.copy()

    df[model_col] = pd.to_numeric(
        df[model_col], errors="coerce"
    )
    df[length_col] = pd.to_numeric(
        df[length_col], errors="coerce"
    )

    valid = (
        df[model_col].notna()
        & df[length_col].notna()
        & np.isfinite(df[model_col])
        & np.isfinite(df[length_col])
    )

    x = df.loc[valid, length_col]
    y = df.loc[valid, model_col]

    pearson = pearsonr(x, y)
    spearman = spearmanr(x, y)

    return {
        "n": int(valid.sum()),
        "pearson_r": pearson.statistic,
        "pearson_p": pearson.pvalue,
        "spearman_rho": spearman.statistic,
        "spearman_p": spearman.pvalue,
    }
corr_with_word_count_sentence(sentence_df_resid)

NameError: name 'sentence_df_resid' is not defined

In [21]:
def corr_with_word_count_person(df, model_name, metric_suffix="loss_resid_median"):
    from scipy.stats import pearsonr, spearmanr
    import numpy as np
    import pandas as pd

    metric = f"{model_name}_{metric_suffix}"

    word_vars = [
        f"{model_name}_total_words",
        f"{model_name}_avg_words",
        f"{model_name}_n_sentences",
    ]

    rows = []

    for word_var in word_vars:
        tmp = df[[metric, word_var]].copy()
        tmp[metric] = pd.to_numeric(tmp[metric], errors="coerce")
        tmp[word_var] = pd.to_numeric(tmp[word_var], errors="coerce")
        tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna()

        if len(tmp) < 3:
            continue

        pearson = pearsonr(tmp[word_var], tmp[metric])
        spearman = spearmanr(tmp[word_var], tmp[metric])

        rows.append(
            {
                "metric": metric,
                "word_var": word_var,
                "n": len(tmp),
                "pearson_r": pearson.statistic,
                "pearson_p": pearson.pvalue,
                "spearman_rho": spearman.statistic,
                "spearman_p": spearman.pvalue,
            }
        )

    return pd.DataFrame(rows)

In [44]:
corr_with_word_count_person(df, MODEL)

,metric,word_var,n,pearson_r,pearson_p,spearman_rho,spearman_p
0,qwen_loss_resid_median,qwen_total_words,366,0.103898,4.700332e-02,0.157802,2.465158e-03
1,qwen_loss_resid_median,qwen_avg_words,366,0.441705,6.545181e-19,0.349939,5.543441e-12
2,qwen_loss_resid_median,qwen_n_sentences,366,-0.223729,1.557278e-05,-0.221442,1.910208e-05


In [44]:
from statsmodels.stats.multitest import multipletests
summary = out["summary"]
# summary is your contrast-level table (one row per contrast — i.e. drop duplicates first)
summary_unique = summary.drop_duplicates(subset="contrast")[["contrast", "lrt_p"]].copy()

reject, p_fdr, _, _ = multipletests(summary_unique["lrt_p"].values, alpha=0.05, method="fdr_bh")
summary_unique["lrt_p_fdr"] = p_fdr
summary_unique["lrt_sig_fdr05"] = reject
print(summary_unique.sort_values("lrt_p"))

      contrast     lrt_p  lrt_p_fdr  lrt_sig_fdr05
9    HC_vs_SSD  0.003588   0.025118           True
18  MDD_vs_SSD  0.014224   0.049785           True
0    HC_vs_all  0.051696   0.120623          False
15   BD_vs_SSD  0.081766   0.143090          False
6    HC_vs_MDD  0.429628   0.518174          False
3     HC_vs_BD  0.444150   0.518174          False
12   BD_vs_MDD  0.747780   0.747780          False


In [3]:
sentence_df = pd.read_csv(SENTENCE_CSV)
metadata = load_metadata(METADATA_CSV)

sentence_df_resid_loss, loss_length_model = add_length_residualized_loss(sentence_df)
sentence_df_resid_ppl, ppl_length_model = add_length_residualized_perplexity(sentence_df_resid_loss)

agg_loss = aggregate_loss_completion(sentence_df_resid_ppl, MODEL)
agg_resid_loss = aggregate_residualized_loss(sentence_df_resid_ppl, MODEL)
agg_ppl = aggregate_perplexity(sentence_df_resid_ppl, MODEL)
agg_resid_ppl = aggregate_residualized_perplexity(sentence_df_resid_ppl, MODEL)

merge_keys = [
    "file_id",
    f"{MODEL}_total_words",
    f"{MODEL}_avg_words",
    f"{MODEL}_n_sentences",
]

person_df = (
    agg_loss
    .merge(agg_resid_loss, on=merge_keys, how="outer")
    .merge(agg_ppl, on=merge_keys, how="outer")
    .merge(agg_resid_ppl, on=merge_keys, how="outer")
)

person_df = add_winsorized_columns(
    person_df,
    cols=[
        f"{MODEL}_loss_resid_median",
        f"{MODEL}_ppl_resid_median",
        f"{MODEL}_ppl_median",
    ],
)

df = metadata.merge(person_df, on="file_id", how="inner")

In [4]:
ppl_vars = [f"{MODEL}_ppl_resid_median_wins"]

In [ ]:
settings = {}
settings["with_outliers"] = df.copy()
settings["without_outliers"] = winsorize_and_log(df, loss_cols) if loss_cols else df.copy()
settings["matched"] = create_matched_sample(df, match_vars=MATCH_VARS)
settings["matched_without_outliers"] = winsorize_and_log(settings["matched"], [c for c in loss_cols if c in settings["matched"].columns]) if loss_cols else settings["matched"].copy()
settings["matched_per_dx"] = create_matched_sample_per_diagnosis(df, match_vars=MATCH_VARS, k=MATCH_K)
settings["matched_per_dx_without_outliers"] = winsorize_and_log(settings["matched_per_dx"], [c for c in loss_cols if c in settings["matched_per_dx"].columns]) if loss_cols else settings["matched_per_dx"].copy()

for name, sdf in settings.items():
    n_pat = (sdf["is_patient"] == 1).sum()
    n_hc = (sdf["is_patient"] == 0).sum()
    print(f"{name}: {len(sdf)} rows ({n_pat} patients, {n_hc} controls)")

  Matched sample: 158 patients + 158 HC  (match_vars=['qwen_avg_words', 'education', 'age', 'sex'])
  BD: 22 patients × up to 2 HCs → 44 matched HC rows (44 unique)
  MDD: 83 patients × up to 2 HCs → 162 matched HC rows (162 unique)
  SSD: 39 patients × up to 2 HCs → 77 matched HC rows (77 unique)
  Per-diagnosis matched total: 144 patients + 283 HC rows  (match_vars=['qwen_avg_words', 'education', 'age', 'sex'], k=2)
with_outliers: 366 rows (189 patients, 177 controls)
without_outliers: 366 rows (189 patients, 177 controls)
matched: 316 rows (158 patients, 158 controls)
matched_without_outliers: 316 rows (158 patients, 158 controls)
matched_per_dx: 427 rows (144 patients, 283 controls)
matched_per_dx_without_outliers: 427 rows (144 patients, 283 controls)


In [20]:
out = perplexity_value_added_by_diagnosis(
        df=settings["without_outliers"],
        include_all_patients=True,
        include_hc_vs_each=True,
        include_patient_pairwise=True,
        confound_vars=["education", "sex_num", "age"],
        length_vars=["qwen_total_words"],
        ppl_vars=ppl_vars,
        n_splits=5,                  # default; auto-drops to small_n_splits when n_min < 25
        n_repeats=20,
        min_n_per_group=10,
        small_class_threshold=25,
        small_n_splits=3,
    )
print(out["summary"].to_string(index=False))


=== HC_vs_all: n_class0=177, n_class1=189 ===  [folds=5]

=== HC_vs_BD: n_class0=177, n_class1=25 ===  [folds=5]

=== HC_vs_MDD: n_class0=177, n_class1=117 ===  [folds=5]

=== HC_vs_SSD: n_class0=177, n_class1=47 ===  [folds=5]

=== BD_vs_MDD: n_class0=25, n_class1=117 ===  [folds=5]

=== BD_vs_SSD: n_class0=25, n_class1=47 ===  [folds=5]

=== MDD_vs_SSD: n_class0=117, n_class1=47 ===  [folds=5]
  contrast  n_class0  n_class1  n_splits       model  mean_auc  auc_ci_lo  auc_ci_hi  delta_auc  delta_ci_lo   delta_ci_hi  delta_p_1s    lrt_p  pr2_baseline  pr2_full  rf_delta_auc
 HC_vs_all       177       189         5 length_only  0.632183   0.505719   0.761091  -0.004654    -0.019685  9.382766e-03      0.7330 0.306713      0.050315  0.052471      0.000020
 HC_vs_all       177       189         5    ppl_only  0.630680   0.506924   0.762588  -0.004654    -0.019685  9.382766e-03      0.7330 0.306713      0.050315  0.052471      0.000020
 HC_vs_all       177       189         5    combined  

In [22]:
corr_with_word_count_person(df, MODEL, metric_suffix="ppl_resid_median")

,metric,word_var,n,pearson_r,pearson_p,spearman_rho,spearman_p
0,qwen_ppl_resid_median,qwen_total_words,366,0.359102,1.398445e-12,0.413778,1.430320e-16
1,qwen_ppl_resid_median,qwen_avg_words,366,0.895678,3.685476e-130,0.921586,1.153391e-151
2,qwen_ppl_resid_median,qwen_n_sentences,366,-0.534668,1.894641e-28,-0.506301,3.186920e-25


In [26]:
sentence_df_resid_ppl

,file_id,task,utterance,word_count,prompt,prompt_condition,completion,full_texts,prompts,completions,perplexity_full,perplexity_completion,avg_loss_full,avg_loss_completion,word_count_full,word_count_completion,log_word_count_completion,avg_loss_length_resid,ppl_length_resid
0,0006_3,1,"Also, hier sehen wir nen Jungen mit nem Instru...",30,Eine psychisch gesunde Person beschreibt ein T...,healthy,"Also, hier sehen wir nen Jungen mit nem Instru...",Eine psychisch gesunde Person beschreibt ein T...,Eine psychisch gesunde Person beschreibt ein T...,"Also, hier sehen wir nen Jungen mit nem Instru...",12.045948,11.803344,2.488728,2.468383,48,30,3.433987,-0.815432,192.930130
1,0006_3,1,"Daneben liegt, wenn ich das richtig sehe, der ...",22,Eine psychisch gesunde Person beschreibt ein T...,healthy,"Daneben liegt, wenn ich das richtig sehe, der ...",Eine psychisch gesunde Person beschreibt ein T...,Eine psychisch gesunde Person beschreibt ein T...,"Daneben liegt, wenn ich das richtig sehe, der ...",19.127822,41.672834,2.951144,3.729849,40,22,3.135494,0.220809,78.672298
2,0006_3,1,Vielleicht hat er das repariert oder daran rum...,8,Eine psychisch gesunde Person beschreibt ein T...,healthy,Vielleicht hat er das repariert oder daran rum...,Eine psychisch gesunde Person beschreibt ein T...,Eine psychisch gesunde Person beschreibt ein T...,Vielleicht hat er das repariert oder daran rum...,15.355163,35.453799,2.731452,3.568230,26,8,2.197225,-0.648774,-380.590175
3,0006_3,1,"Ich bin mir nicht ganz sicher, aber ich vermut...",19,Eine psychisch gesunde Person beschreibt ein T...,healthy,"Ich bin mir nicht ganz sicher, aber ich vermut...",Eine psychisch gesunde Person beschreibt ein T...,Eine psychisch gesunde Person beschreibt ein T...,"Ich bin mir nicht ganz sicher, aber ich vermut...",17.029621,30.685948,2.834954,3.423805,37,19,2.995732,-0.190692,0.201367
4,0006_3,1,Auf jeden Fall scheint er damit aber irgendwie...,45,Eine psychisch gesunde Person beschreibt ein T...,healthy,Auf jeden Fall scheint er damit aber irgendwie...,Eine psychisch gesunde Person beschreibt ein T...,Eine psychisch gesunde Person beschreibt ein T...,Auf jeden Fall scheint er damit aber irgendwie...,15.431969,18.981522,2.736441,2.943466,63,45,3.828641,-0.042566,390.667061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26925,﻿1215_2,6,So mehr will ich und brauche ich gar nicht sag...,17,Eine psychisch gesunde Person beschreibt ein T...,healthy,So mehr will ich und brauche ich gar nicht sag...,Eine psychisch gesunde Person beschreibt ein T...,Eine psychisch gesunde Person beschreibt ein T...,So mehr will ich und brauche ich gar nicht sag...,23.070614,91.064035,3.138560,4.511563,35,17,2.890372,0.817567,9.706135
26926,﻿1215_2,6,Polnischer Schauspieler Horst Buchholz könnte ...,18,Eine psychisch gesunde Person beschreibt ein T...,healthy,Polnischer Schauspieler Horst Buchholz könnte ...,Eine psychisch gesunde Person beschreibt ein T...,Eine psychisch gesunde Person beschreibt ein T...,Polnischer Schauspieler Horst Buchholz könnte ...,17.294155,31.579901,2.850369,3.452521,36,18,2.944439,-0.200679,-23.671644
26927,﻿1215_2,6,"Ja, die Mutter umarmt ihren Sohn und nimmt ihn...",34,Eine psychisch gesunde Person beschreibt ein T...,healthy,"Ja, die Mutter umarmt ihren Sohn und nimmt ihn...",Eine psychisch gesunde Person beschreibt ein T...,Eine psychisch gesunde Person beschreibt ein T...,"Ja, die Mutter umarmt ihren Sohn und nimmt ihn...",14.183496,17.275524,2.652079,2.849291,52,34,3.555348,-0.342953,257.001392
26928,﻿1215_2,6,Das ist offen.,3,Eine psychisch gesunde Person beschreibt ein T...,healthy,Das ist offen.,Eine psychisch gesunde Person beschreibt ein T...,Eine psychisch gesunde Person beschreibt ein T...,Das ist offen.,14.806445,456.787088,2.695063,6.124217,21,3,1.386294,1.295332,-350.814494


In [27]:
corr_with_word_count_sentence(sentence_df_resid_ppl, model_col="ppl_length_resid", length_col="word_count_completion")

{'n': 26930,
 'pearson_r': np.float64(0.02496096540584223),
 'pearson_p': np.float64(4.192026678323666e-05),
 'spearman_rho': np.float64(0.7758822439393238),
 'spearman_p': np.float64(0.0)}